In [1]:
import pandas as pd

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
books = pd.read_csv("data/processed/books_with_genres.csv")
books["isbn13"] = books["isbn13"].astype(str)

In [3]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

In [4]:
vector_store = Chroma(
    persist_directory="chroma_db",
    embedding_function=embedding_model
)

In [5]:
def retrieve_books(query, k=30):
    results = vector_store.similarity_search(query, k=k)
    return results

In [6]:
def unique_books(results):
    seen = set()
    unique = []

    for doc in results:
        isbn = doc.metadata["isbn13"]

        if isbn not in seen:
            seen.add(isbn)
            unique.append(doc)

    return unique

In [7]:
def documents_to_dataframe(documents):
    isbn_order = [doc.metadata["isbn13"] for doc in documents]

    df = books[books["isbn13"].isin(isbn_order)].copy()

    df["isbn13"] = pd.Categorical(
        df["isbn13"],
        categories=isbn_order,
        ordered=True
    )

    df = df.sort_values("isbn13").reset_index(drop=True)

    return df

In [8]:
books["isbn13"] = books["isbn13"].astype(str)

In [9]:
def filter_books(
    df,
    genre="All",
    emotion="Any",
    min_rating=0
):
    filtered = df.copy()

    if genre != "All":
        filtered = filtered[
            filtered["display_genre"] == genre
        ]

    if emotion != "Any":
        filtered = filtered[
            filtered["dominant_emotion"] == emotion
        ]

    filtered = filtered[
        filtered["average_rating"] >= min_rating
    ]

    return filtered

In [10]:
def recommend_books(
    query,
    genre="All",
    emotion="Any",
    min_rating=0,
    top_k=5
):
    # Step 1: Retrieve from Chroma
    results = retrieve_books(query, k=50)

    # Step 2: Remove duplicate books
    results = unique_books(results)

    # Step 3: Convert to DataFrame
    df = documents_to_dataframe(results)

    # Step 4: Apply filters
    df = filter_books(
        df,
        genre=genre,
        emotion=emotion,
        min_rating=min_rating
    )

    # Step 5: No books found
    if df.empty:
        return df

    # Step 6: Return top books
    return (
        df.sort_values(
            by="average_rating",
            ascending=False
        )
        .head(top_k)[
            [
                "title",
                "authors",
                "display_genre",
                "average_rating",
                "dominant_emotion",
                "thumbnail",
                "description",
            ]
        ]
    )

In [11]:
import gradio as gr

In [12]:
def format_results(df, explanations):

    if len(df) == 0:
        return """
        <div style="
            padding:80px;
            text-align:center;
            color:white;
            font-size:26px;
            background:#111;
            border-radius:24px;
            border:1px solid #222;
        ">
            😔 No books found.
        </div>
        """

    html = ""

    for i, (_, row) in enumerate(df.iterrows()):

        description = str(row["description"])[:320]

        html += f"""

        <div class="book-card" style="

            display:flex;
            gap:22px;

            background:#111111;

            border:1px solid #2d2d2d;

            border-radius:28px;

            padding:22px;

            margin:20px 0;

            box-shadow:
            0 18px 55px rgba(0,0,0,.45);

        ">

            <img

                src="{row['thumbnail']}"

                style="

                width:145px;

                height:215px;

                border-radius:18px;

                object-fit:cover;

                box-shadow:
                0 18px 40px rgba(0,0,0,.45);

                flex-shrink:0;

            ">

            <div style="flex:1;">

                <div style="

                    font-size:28px;

                    color:white;

                    font-weight:700;

                    line-height:1.25;

                    margin-bottom:10px;

                ">

                    {row['title']}

                </div>

                <div style="

                    color:#9ca3af;

                    font-size:19px;

                    margin-bottom:22px;

                ">

                    {row['authors']}

                </div>

                <div style="

                    display:flex;

                    gap:12px;

                    flex-wrap:wrap;

                    margin-bottom:24px;

                ">

                    <span style="

                        background:#f59e0b;

                        color:black;

                        padding:9px 18px;

                        border-radius:999px;

                        font-weight:700;

                    ">

                        ⭐ {row['average_rating']}

                    </span>

                    <span style="

                        background:#2563eb;

                        color:white;

                        padding:9px 18px;

                        border-radius:999px;

                    ">

                        📚 {row['display_genre']}

                    </span>

                    <span style="

                        background:#059669;

                        color:white;

                        padding:9px 18px;

                        border-radius:999px;

                    ">

                        😊 {row['dominant_emotion']}

                    </span>

                </div>

                <div style="

                    color:#d4d4d8;

                    font-size:17px;

                    line-height:1.9;

                    margin-bottom:28px;

                ">

                    {description}...

                </div>

                <div style="

                    background:linear-gradient(
                    180deg,
                    #1e293b,
                    #172033
                    );

                    border-left:5px solid #38bdf8;

                    border-radius:18px;

                    padding:20px;

                ">

                    <div style="

                        color:white;

                        font-size:18px;

                        font-weight:700;

                        margin-bottom:12px;

                    ">

                        💡 Why this book?

                    </div>

                    <div style="

                        color:#d6d6d6;

                        font-size:15px;

                        line-height:1.6;

                    ">

                        {explanations[i]}

                    </div>

                </div>

            </div>

        </div>

        """

    return html

In [13]:
def search_books(query, genre, emotion, rating):

    results = recommend_books(
        query=query,
        genre=genre,
        emotion=emotion,
        min_rating=rating,
        top_k=5
    )

    if len(results) == 0:
        return """
        <div style="
            text-align:center;
            padding:60px;
            color:white;
            font-size:24px;
        ">
            😔 No books found matching your preferences.
        </div>
        """

    # AI Summary
    summary, avg_rating, top_genre = generate_ai_summary(
        query=query,
        genre=genre,
        emotion=emotion,
        rating=rating,
        books_df=results
    )

    # AI explanation for each book
    explanations = generate_book_explanations(
        query=query,
        genre=genre,
        emotion=emotion,
        rating=rating,
        books_df=results
    )

    html = f"""
    

    <div style="
        background:#111111;
        color:white;
        padding:30px;
        border-radius:22px;
        margin-bottom:35px;
        border:1px solid #222;
        box-shadow:0 12px 35px rgba(0,0,0,.45);
    ">

        <h2 style="
            margin-top:0;
            margin-bottom:18px;
            font-size:30px;
        ">
            🤖 AI Recommendation
        </h2>

        <p style="
            font-size:18px;
            color:#d6d6d6;
            line-height:1.8;
            margin-bottom:25px;
        ">
            {summary}
        </p>

        <div style="
            display:flex;
            gap:15px;
            flex-wrap:wrap;
        ">

            <span style="
                background:#f59e0b;
                color:black;
                padding:10px 18px;
                border-radius:999px;
                font-weight:bold;
            ">
                ⭐ Average Rating: {avg_rating}
            </span>

            <span style="
                background:#2563eb;
                color:white;
                padding:10px 18px;
                border-radius:999px;
            ">
                📚 {top_genre}
            </span>

            <span style="
                background:#10b981;
                color:white;
                padding:10px 18px;
                border-radius:999px;
            ">
                📖 {len(results)} Books
            </span>

        </div>

    </div>

    """

    html += format_results(
        results,
        explanations
    )

    return html

In [14]:
css = """

@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');


/* =====================================
   GLOBAL
===================================== */

html,
body{

    margin:0;
    padding:0;

    font-family:'Inter',sans-serif;

    color:white;

    overflow-x:hidden;

}




/* =====================================
   CONTAINER
===================================== */

.gradio-container{

    position:relative;

    z-index:5;

}


/* Remove default backgrounds */

.gr-block,
.gr-box,
.gr-form,
.gr-group{

    background:transparent !important;

    border:none !important;

    box-shadow:none !important;

}


/* =====================================
   HERO
===================================== */

.hero{

    text-align:center;

    padding:55px 35px;

    margin-bottom:30px;

    border-radius:24px;

    background:rgba(18,18,18,.85);

    border:1px solid rgba(255,255,255,.08);

    backdrop-filter:blur(10px);

}


.hero h1{

    font-size:56px;

    margin:0;

    font-weight:700;

    letter-spacing:-1px;

}


.hero p{

    margin-top:16px;

    color:#bdbdbd;

    font-size:20px;

    line-height:1.7;

}


/* =====================================
   SEARCH PANEL
===================================== */

.search-panel{

background:rgba(18,18,18,.88);

backdrop-filter:blur(10px);

}

/* =====================================
   INPUTS
===================================== */

textarea,
input{

    background:#181818 !important;

    color:white !important;

    border:1px solid #2d2d2d !important;

    border-radius:14px !important;

    font-size:15px !important;

}


textarea:focus,
input:focus{

    border:1px solid #666 !important;

    box-shadow:none !important;

}


/* =====================================
   DROPDOWNS
===================================== */

.gr-dropdown{

    border-radius:14px !important;

}


/* =====================================
   LABELS
===================================== */

.gr-form label{

    color:#d5d5d5 !important;

    font-size:15px !important;

    font-weight:600 !important;

}


/* =====================================
   BUTTON
===================================== */

button{

    background:#4f46e5 !important;

    color:white !important;

    border:none !important;

    border-radius:14px !important;

    font-size:17px !important;

    font-weight:600 !important;

    padding:15px !important;

    transition:.25s;

}


button:hover{

    background:#6558ff !important;

    transform:translateY(-2px);

}


/* =====================================
   SLIDER
===================================== */

.gr-slider{

    padding-top:6px;

}
/* =====================================
   BOOK CARDS
===================================== */

.book-card{

    background:rgba(17,17,17,.92);

    border:1px solid rgba(255,255,255,.08);

    border-radius:24px;

    overflow:hidden;

    transition:all .25s ease;

}

.book-card:hover{

    transform:translateY(-4px);

    box-shadow:0 18px 45px rgba(0,0,0,.45);

}


/* =====================================
   BADGES
===================================== */

.badge{

    display:inline-flex;

    align-items:center;

    padding:7px 15px;

    border-radius:999px;

    font-size:14px;

    font-weight:600;

}


.badge-rating{

    background:#d97706;

    color:white;

}

.badge-genre{

    background:#262626;

    color:white;

}

.badge-emotion{

    background:#262626;

    color:white;

}


/* =====================================
   AI EXPLANATION BOX
===================================== */

.ai-box{

    margin-top:18px;

    padding:18px;

    border-left:4px solid #4f46e5;

    border-radius:12px;

    background:#171717;

}

.ai-title{

    font-size:17px;

    font-weight:700;

    color:white;

    margin-bottom:10px;

}

.ai-text{

    color:#d1d5db;

    line-height:1.7;

    font-size:15px;

}


/* =====================================
   HTML OUTPUT
===================================== */

.gr-html{

    width:100% !important;

}


/* =====================================
   LINKS
===================================== */

a{

    color:#9db8ff;

    text-decoration:none;

}

a:hover{

    color:white;

}


/* =====================================
   FOOTER
===================================== */

footer{

    display:none !important;

}


/* =====================================
   SCROLLBAR
===================================== */

::-webkit-scrollbar{

    width:10px;

}

::-webkit-scrollbar-track{

    background:#090909;

}

::-webkit-scrollbar-thumb{

    background:#3a3a3a;

    border-radius:50px;

}

::-webkit-scrollbar-thumb:hover{

    background:#555;

}


/* =====================================
   MOBILE
===================================== */

@media(max-width:900px){

.gradio-container{

    padding:18px !important;

}

.hero{

    padding:35px 20px;

}

.hero h1{

    font-size:38px;

}

.hero p{

    font-size:16px;

}

.book-card{

    flex-direction:column !important;

}

.book-card img{

    width:180px !important;

    height:auto !important;

    margin:auto;

}

}


/* =====================================
   REMOVE BLUE GRADIO COLORS
===================================== */

.gr-button-primary{

    background:#4f46e5 !important;

}

.gr-button-primary:hover{

    background:#6558ff !important;

}
.hero{

backdrop-filter:blur(12px);

background:rgba(10,10,10,.82);

}

.hero h1{

    font-size:64px;

    color:white;

    font-weight:700;

    margin-bottom:18px;

}

.hero p{

    color:#bdbdbd;

    font-size:22px;

}

/* ==========================
   WELCOME CARD
========================== */

.welcome-card{

    background:#111111;

    border:1px solid #242424;

    border-radius:24px;

    padding:45px;

    margin-top:35px;

    color:white;

    text-align:center;

    box-shadow:0 20px 60px rgba(0,0,0,.45);

}

.welcome-icon{

    font-size:55px;

    margin-bottom:10px;

}

.welcome-card h2{

    margin:10px 0;

    font-size:34px;

    font-weight:700;

}

.welcome-card>p{

    color:#a1a1aa;

    font-size:18px;

    max-width:750px;

    margin:0 auto 35px;

    line-height:1.7;

}

.feature-grid{

    display:grid;

    grid-template-columns:repeat(2,1fr);

    gap:20px;

    margin-top:20px;

}

.feature{

    background:#171717;

    border:1px solid #2d2d2d;

    border-radius:18px;

    padding:22px;

    display:flex;

    gap:18px;

    align-items:flex-start;

    text-align:left;

}

.feature span{

    font-size:32px;

}

.feature h4{

    margin:0;

    color:white;

    font-size:18px;

}

.feature p{

    margin:5px 0 0;

    color:#9ca3af;

    font-size:15px;

}

.start-tip{

    margin-top:35px;

    padding:18px;

    border-radius:15px;

    background:#1b1b1b;

    color:#d4d4d4;

    font-size:17px;

}

@media(max-width:900px){

.feature-grid{

    grid-template-columns:1fr;

}

}
.gr-box,
.gr-group,
.gr-form{

    background:transparent !important;

}


/* =====================================
   END
===================================== */

"""

In [15]:


with gr.Blocks(
    title="StoryVerse",
    css=css,
    theme=gr.themes.Base()
) as demo:

    gr.HTML("""

""")
    gr.HTML("""
            
<div class="hero">

    <h1>📚 StoryVerse</h1>

    <p>
        Discover your next favorite book using AI-powered semantic search.
    </p>

</div>
""")

    with gr.Row():

        query = gr.Textbox(
            label="Search",
            placeholder="e.g. books about friendship"
        )

        genre = gr.Dropdown(
            choices=["All"] + sorted(books["display_genre"].unique().tolist()),
            value="All",
            label="Genre"
        )

        emotion = gr.Dropdown(
            choices=[
                "Any",
                "joy",
                "fear",
                "anger",
                "sadness",
                "neutral",
                "surprise",
                "disgust"
            ],
            value="Any",
            label="Emotion"
        )

        rating = gr.Slider(
            minimum=0,
            maximum=5,
            step=0.5,
            value=0,
            label="Minimum Rating"
        )

    

    button = gr.Button("✨ Discover Books")

    default_output = """
<div class="welcome-card">

    <div class="welcome-icon">
        🤖
    </div>

    <h2>Welcome to StoryVerse</h2>

    <p>
        Discover books using <b>Semantic Search</b>, emotion analysis and AI-powered recommendations.
    </p>

    <div class="feature-grid">

        <div class="feature">
            <span>📚</span>
            <div>
                <h4>7000+ Books</h4>
                <p>Large curated collection</p>
            </div>
        </div>

        <div class="feature">
            <span>🧠</span>
            <div>
                <h4>Semantic Search</h4>
                <p>Find books by meaning</p>
            </div>
        </div>

        <div class="feature">
            <span>😊</span>
            <div>
                <h4>Emotion Filter</h4>
                <p>Search by mood</p>
            </div>
        </div>

        <div class="feature">
            <span>✨</span>
            <div>
                <h4>AI Explanations</h4>
                <p>Know why each book fits</p>
            </div>
        </div>

    </div>

    <div class="start-tip">
        👆 Start by typing a book title, author or describe what you want to read.
    </div>

</div>
"""

    output = gr.HTML(value=default_output)

    button.click(
    fn=search_books,
    inputs=[query, genre, emotion, rating],
    outputs=output
)

    button.click(
        fn=search_books,
        inputs=[query, genre, emotion, rating],
        outputs=output
    )



C:\Users\rakes\AppData\Local\Temp\ipykernel_27596\1837307694.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [16]:
from dotenv import load_dotenv
import os
load_dotenv("../.env")

False

In [17]:
import os

api_key = os.getenv("MISTRAL_API_KEY")


In [18]:
from langchain_mistralai import ChatMistralAI

In [19]:
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=api_key,
    temperature=0.3
)

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [21]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant for a semantic book recommendation system.

User Preferences

Search Query:
{query}

Genre:
{genre}

Emotion:
{emotion}

Minimum Rating:
{rating}

Retrieved Books:
{books}

Instructions:
- Explain why the retrieved books are good recommendations.
- Treat the books as one collection.
- Never mention any book title.
- Never mention any author.
- Never invent books or information.
- Use ONLY the retrieved books.
- Mention the selected genre or emotion only if relevant.
- Keep the response between 60 and 80 words.
- Write one engaging paragraph.

Recommendation Summary:
""")

In [22]:
parser = StrOutputParser()

In [23]:
chain = prompt | llm | parser

In [24]:
def generate_ai_summary(
    query,
    genre,
    emotion,
    rating,
    books_df
):

    books_text = ""

    for _, row in books_df.iterrows():

        books_text += f"""
Title: {row['title']}
Genre: {row['display_genre']}
Rating: {row['average_rating']}
Description: {row['description'][:250]}

"""

    summary = chain.invoke(
        {
            "query": query if query else "None",
            "genre": genre,
            "emotion": emotion,
            "rating": rating,
            "books": books_text
        }
    )

    avg_rating = round(
        books_df["average_rating"].mean(),
        2
    )

    top_genre = books_df["display_genre"].mode()[0]

    return summary, avg_rating, top_genre

In [25]:
explanation_prompt = ChatPromptTemplate.from_template("""
You are an AI assistant for a book recommendation system.

User Preferences

Search Query: {query}

Genre: {genre}

Emotion: {emotion}

Minimum Rating: {rating}

Books:

{books}

Instructions:
- Write ONE short explanation for each book.
- Maximum 15 words.
- Do NOT start with the book title.
- Do NOT repeat the user's query.
- Make each explanation unique.
- Use only the provided information.
- Return one explanation per line.
""")

In [26]:
explanation_chain = explanation_prompt | llm | parser

In [27]:
def generate_book_explanations(
    query,
    genre,
    emotion,
    rating,
    books_df
):

    books_text = ""

    for _, row in books_df.iterrows():

        books_text += f"""
Title: {row['title']}
Genre: {row['display_genre']}
Rating: {row['average_rating']}
Description: {row['description'][:200]}

"""

    response = explanation_chain.invoke(
        {
            "query": query if query else "None",
            "genre": genre,
            "emotion": emotion,
            "rating": rating,
            "books": books_text
        }
    )

    explanations = [
        line.strip("-•1234567890. ")
        for line in response.splitlines()
        if line.strip()
    ]

    # Safety check
    while len(explanations) < len(books_df):
        explanations.append("A strong recommendation based on your selected preferences.")

    return explanations[:len(books_df)]

In [28]:
recommended_books = recommend_books(
    "fantasy books",
    "Fantasy",
    "Any",
    4,
    5
)

generate_book_explanations(
    "fantasy books",
    "Fantasy",
    "Any",
    4,
    recommended_books
)

['Magical adventures await in a world of wizards and spells',
 'A magical saga spanning six years of wonder and friendship',
 'Epic quest to destroy a powerful ring in a richly detailed world',
 "Young hobbit's journey begins in this classic fantasy adventure",
 'Time-traveling siblings explore science and magic in a timeless tale']

In [29]:
import os

print(os.getcwd())
print(os.path.exists("26380884.jpg"))

c:\Users\rakes\Desktop\book recommender
False


In [ ]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\rakes\Desktop\book recommender\env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\rakes\Desktop\book recommender\env\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
